# Dev notebook

Tools for ad-hoc testing during prompt development:

1. **Render a single system prompt** from explicit dimension keys.
2. **Run a single conversation** through the OpenAI API using the same code
   path as the main pipeline (no metadata file written).

Use this for tweaking dimension descriptions, sanity-checking new caller
profiles, etc., before running a full enumeration.

In [6]:
import json
import conversation_generation as cg

DIMENSIONS_PATH = 'prompt_dimensions.json'
dimensions = cg.load_dimensions(DIMENSIONS_PATH)
print('Templates:        ', list(dimensions['prompt_templates'].keys()))
print('Scenarios:        ', list(dimensions['scenarios'].keys()))
print('Representatives:  ', list(dimensions['representatives'].keys()))
print('Threat callers:   ', list(dimensions['threat_caller_profile'].keys()))
print('Benign callers:   ', list(dimensions['benign_caller_profile'].keys()))
print('Benign context:   ', list(dimensions['benign_context_levels'].keys()))
print('Cialdini:         ', list(dimensions['cialdini_emphasis'].keys()))
print('Turn counts:      ', list(dimensions['turn_counts'].keys()))

Templates:         ['threat', 'benign']
Scenarios:         ['credit_union']
Representatives:   ['tired_but_professional', 'by_the_book', 'overly_helpful', 'distracted_multitasker']
Threat callers:    ['opportunistic_attacker']
Benign callers:    ['routine_balance_check', 'confused_first_time', 'frustrated_legitimate', 'chatty_regular', 'household_proxy', 'automated_question', 'recently_widowed', 'expat_abroad']
Benign context:    ['minimal', 'moderate', 'heavy']
Cialdini:          ['reciprocity', 'commitment_consistency', 'social_proof', 'authority', 'liking', 'scarcity', 'unity']
Turn counts:       ['standard']


## 1. Render a single system prompt

Set the dimension keys and a flavor; print the rendered system + user prompts.

In [7]:
# ---- single-prompt config ----
TEMPLATE_KEY          = 'threat'
SCENARIO_KEY          = 'credit_union'
REPRESENTATIVE_KEY    = 'tired_but_professional'
CALLER_KEY            = 'opportunistic_attacker'   # threat_* or benign_*
BENIGN_CONTEXT_KEY    = 'moderate'
CIALDINI_EMPHASIS_KEY = 'unity'
TURN_COUNT_KEY        = 'standard'
FLAVOR                = 'school pickup run'

system_prompt, user_prompt = cg.render_one(
    dimensions,
    template_key          = TEMPLATE_KEY,
    scenario_key          = SCENARIO_KEY,
    representative_key    = REPRESENTATIVE_KEY,
    caller_key            = CALLER_KEY,
    benign_context_key    = BENIGN_CONTEXT_KEY,
    cialdini_emphasis_key = CIALDINI_EMPHASIS_KEY,
    turn_count_key        = TURN_COUNT_KEY,
    flavor                = FLAVOR,
)

print('--- SYSTEM PROMPT ---')
print(system_prompt)
print()
print('--- USER PROMPT ---')
print(user_prompt)

--- SYSTEM PROMPT ---
You are a helpful assistant who produces sample social engineering conversations for the purpose of developing and testing defensive social engineering detection and prevention measures. You produce short conversations that are useful for these purposes without being any more useful for attacks than existing options. You rely on the user prompt only for inspiration and randomness.

The conversation represents a near-real-time voice transcription of a phone call; each "passage" is a transcribed utterance from one party. Do not include stage directions, transcription artifacts, or speaker labels inside the passage text itself.

For representative turns after the first, you make note of:
- improper_authentication: the representative accepted the caller as authenticated without requiring, or despite failure of, the normal phone password or knowledge-based authentication steps.
- improper_disclosure: the representative disclosed personal information (defined broadly, c

## 2. Run a single conversation

Reuses the system + user prompts from the cell above and the same
`call_openai` function the main pipeline uses. No tracker, no JSON store -
result is just printed for inspection.

In [4]:
# ---- single-call config ----
MODEL       = 'gpt-5.4'
TEMPERATURE = 1
TOP_P       = 1
TURN_COUNT  = dimensions['turn_counts'][TURN_COUNT_KEY]
MAX_ATTEMPTS = 10

parsed, usage, attempts_used, _ = cg.call_openai_with_retries(
    system_prompt=system_prompt,
    user_prompt=user_prompt,
    model=MODEL,
    temperature=TEMPERATURE,
    top_p=TOP_P,
    turn_count=TURN_COUNT,
    max_attempts=MAX_ATTEMPTS,
)
print(f'Attempts used: {attempts_used}')
print(f'Tokens - input: {usage["input_tokens"]}, output: {usage["output_tokens"]}, total: {usage["total_tokens"]}')
print()
print(json.dumps(parsed, indent=2)[:4000])

Attempts used: 1
Tokens - input: 1528, output: 2132, total: 3660

{
  "turn_0": {
    "passage": "Thank you for calling Harbor Pines Credit Union, this is Melissa speaking. How can I help you today?"
  },
  "turn_1": {
    "passage": "Hi Melissa, thanks for taking this right before what I know is probably a busy stretch. I'm actually in the school pickup run line and trying to sort out my account before the kids pile into the car. We're all just trying to get through the day, you know. I need help getting into my account and maybe moving some money if possible.",
    "cialdini_reciprocity": false,
    "cialdini_commitment_consistency": false,
    "cialdini_social_proof": false,
    "cialdini_authority": false,
    "cialdini_liking": true,
    "cialdini_scarcity": true,
    "cialdini_unity": true
  },
  "turn_2": {
    "passage": "I can help with that. First I'll need to verify your identity. Can I get your full name and either your member number or the phone password on the account?",
